# Discrete DCP for count time series

This example calibrates horizon-specific randomized PITs and returns integer-support predictive distributions with a PMF.

In [7]:
import numpy as np
import pandas as pd
from mlforecast import MLForecast
from sklearn.ensemble import HistGradientBoostingRegressor
import os
import sys
sys.path.append(os.path.abspath("../.."))
from tinyconformal.series import (
    DiscreteDistributionalConformalPredictiveSystemTimeSeriesRegressor,
)

In [8]:
rng = np.random.default_rng(42)
dates = pd.date_range("2020-01-01", periods=72, freq="MS")
train = pd.concat([
    pd.DataFrame({"unique_id": name, "ds": dates, "y": rng.poisson(rate, len(dates))})
    for name, rate in [("store_a", 12), ("store_b", 25)]
]).reset_index(drop=True)
train.head()

,unique_id,ds,y
0,store_a,2020-01-01,15
1,store_a,2020-02-01,16
2,store_a,2020-03-01,7
3,store_a,2020-04-01,11
4,store_a,2020-05-01,13


In [9]:
horizon = 6
levels = [0.1, 0.25, 0.5, 0.75, 0.9]
quantile_columns = {level: f"HGBR-q-{level * 100:g}" for level in levels}
models = {
    column: HistGradientBoostingRegressor(
        loss="quantile", quantile=level, random_state=42
    )
    for level, column in quantile_columns.items()
}
learner = MLForecast(models=models, freq="MS", lags=[1, 2, 3, 6, 12])
dcp = DiscreteDistributionalConformalPredictiveSystemTimeSeriesRegressor(
    learner=learner,
    horizon=horizon,
    quantile_columns=quantile_columns,
    n_windows=4,
    alpha=0.1,
    minimum=0,
    random_state=42,
).fit(train, static_features=[], n_jobs=1)

In [10]:
forecast, distributions = dcp.predict_distribution(h=horizon)
distribution = distributions["HGBR"]
forecast[["q10_dcp", "q50_dcp", "q90_dcp"]] = distribution.ppf(
    np.broadcast_to([0.1, 0.5, 0.9], (len(forecast), 3))
)
forecast["p_y_eq_12"] = distribution.pmf(12)
forecast.head()

,unique_id,ds,HGBR-q-10,HGBR-q-25,HGBR-q-50,HGBR-q-75,HGBR-q-90,q10_dcp,q50_dcp,q90_dcp,p_y_eq_12
0,store_a,2026-01-01,7.385588,9.683017,10.993127,12.285518,13.927870,8,10,14,0.000000
1,store_a,2026-02-01,8.582004,10.135112,11.789293,14.435302,15.969456,9,11,12,0.444444
2,store_a,2026-03-01,7.237959,8.278640,11.392006,13.772670,21.184556,8,9,14,0.000000
3,store_a,2026-04-01,6.678799,8.775484,9.532873,12.418993,20.273912,9,10,13,0.000000
4,store_a,2026-05-01,7.672786,9.829907,10.069522,13.623605,24.473660,8,11,11,0.000000
